In [ ]:
# Mohammed Sufiyan
# U09479155

In [1]:
from pathlib import Path
import sys, os

ROOT = Path('/Users/suff/Documents/New project')
assert ROOT.exists(), f'Project folder not found: {ROOT}'
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.cache' / 'matplotlib'))
print('Working dir:', Path.cwd())
print('ROOT on path:', str(ROOT) in sys.path)


Working dir: /Users/suff/Documents/New project
ROOT on path: True


In [2]:
import sys, subprocess
pkgs = ['stable-baselines3', 'gymnasium', 'mujoco', 'pyyaml', 'tensorboard', 'imageio']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('Dependencies ready')


Dependencies ready


In [3]:
# 3)Import
import yaml
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from robot.generator import load_config
from sim.env import BirdBipedWalkEnv, RewardWeights

print('Imports OK')


Imports OK


In [4]:
# 4)Paths and configuration the xml part you know
CONFIG_PATH = ROOT / 'config' / 'design.yaml'
XML_PATH = ROOT / 'assets' / 'generated' / 'bird_biped.xml'
MODEL_OUT = ROOT / 'checkpoints' / 'ppo_bird_biped_notebook'
LOGDIR = ROOT / 'logs' / 'tb_notebook'

cfg = load_config(CONFIG_PATH)

assert CONFIG_PATH.exists(), CONFIG_PATH
assert XML_PATH.exists(), f'Missing XML: {XML_PATH} (run scripts/generate_robot.py first)'
print('Config:', CONFIG_PATH)
print('XML:', XML_PATH)


Config: /Users/suff/Documents/New project/config/design.yaml
XML: /Users/suff/Documents/New project/assets/generated/bird_biped.xml


In [5]:
#5)Building the environment
rw_cfg = cfg.reward
reward_weights = RewardWeights(
    alive_bonus=rw_cfg.alive_bonus,
    forward_weight=rw_cfg.forward_weight,
    control_cost_weight=rw_cfg.control_cost_weight,
    postural_cost_weight=rw_cfg.postural_cost_weight,
    lateral_cost_weight=rw_cfg.lateral_cost_weight,
)

def make_env():
    env = BirdBipedWalkEnv(
        xml_path=str(XML_PATH),
        frame_skip=cfg.sim.frame_skip,
        episode_seconds=cfg.sim.episode_seconds,
        desired_speed=cfg.sim.desired_speed,
        reward_weights=reward_weights,
    )
    return Monitor(env)

env = DummyVecEnv([make_env])
print('Env ready. action_dim =', env.action_space.shape)


Env ready. action_dim = (6,)


In [6]:
#6)Train PPO
train_cfg = cfg.training
TIMESTEPS = 300000  # change if needed

model = PPO(
    policy='MlpPolicy',
    env=env,
    learning_rate=train_cfg.learning_rate,
    n_steps=train_cfg.n_steps,
    batch_size=train_cfg.batch_size,
    gamma=train_cfg.gamma,
    gae_lambda=train_cfg.gae_lambda,
    clip_range=train_cfg.clip_range,
    ent_coef=train_cfg.ent_coef,
    verbose=1,
    tensorboard_log=str(LOGDIR),
    seed=train_cfg.seed,
)

model.learn(total_timesteps=TIMESTEPS, progress_bar=False)
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
model.save(str(MODEL_OUT))
print('Saved model:', str(MODEL_OUT) + '.zip')


Using cpu device
Logging to /Users/suff/Documents/New project/logs/tb_notebook/PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 12.3     |
|    ep_rew_mean     | 11.8     |
| time/              |          |
|    fps             | 9776     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 12.4        |
|    ep_rew_mean          | 12.1        |
| time/                   |             |
|    fps                  | 7435        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.006728247 |
|    clip_fraction        | 0.0583      |
|    clip_range           | 0.18        |
|    entropy_loss         | -

In [7]:
#7)Quick evaluation
model = PPO.load(str(MODEL_OUT) + '.zip')

eval_env = BirdBipedWalkEnv(
    xml_path=str(XML_PATH),
    frame_skip=cfg.sim.frame_skip,
    episode_seconds=cfg.sim.episode_seconds,
    desired_speed=cfg.sim.desired_speed,
    reward_weights=reward_weights,
)

returns = []
for ep in range(3):
    obs, _ = eval_env.reset()
    done = False
    trunc = False
    ep_ret = 0.0
    while not (done or trunc):
        action, _ = model.predict(obs, deterministic=True)
        obs, r, done, trunc, _ = eval_env.step(action)
        ep_ret += r
    returns.append(ep_ret)
    print(f'Episode {ep+1} return: {ep_ret:.2f}')

eval_env.close()
print('Mean return:', sum(returns)/len(returns))


Episode 1 return: 30.30
Episode 2 return: 30.43
Episode 3 return: 30.56
Mean return: 30.428461515195323
